# 3 - Problema de investimentos em projetos


In [ ]:
import gurobipy as gp
from gurobipy import GRB

In [ ]:
projetos = 6
periodos = 4
I = [i for i in range(projetos)]
J = [j for j in range(periodos)]

In [ ]:
lucro = [32.4, 35.8, 17.75, 14.8, 18.2, 12.35]
recursos = [60, 70, 35, 20]

In [ ]:
desembolso = [[10.5, 14.4, 2.2, 2.4],[8.3, 12.6, 9.5, 3.1],[10.2, 14.2, 5.6, 4.2],
              [7.2, 10.5, 7.5, 5.0],[12.3, 10.1, 8.3, 6.3],[9.2, 7.8, 6.9, 5.1]]
custo_total = [29.5, 33.5, 34.2, 30.2, 37.0, 29.0]

### Caso A


In [ ]:
# Modelo
m = gp.Model("Investimento-a")

# Variáveis de decisão
x = m.addVars (I, vtype = GRB.BINARY, obj = lucro, name = "x")

# Restrições
# Respeitar a disponibilidade de recursos
m.addConstrs(gp.quicksum(x[i]*desembolso[i][j] for i in I) <= recursos[j] for j in J)

m.ModelSense = GRB.MAXIMIZE
m.update()
m.write("Modelo_Investimento-a.lp")
m.optimize()

# Saída...
for i in range(projetos):
    print(f"Projeto {i+1} -> Executar (1-sim, 0-não): {x[i].x}")

### Caso B


In [ ]:
# Modelo
m = gp.Model("Investimento-b")

# Variáveis de decisão
x = m.addVars (I, vtype = GRB.BINARY, obj = lucro, name = "x")

# Restrições
# Respeitar a disponibilidade de recursos
m.addConstrs(gp.quicksum(x[i]*desembolso[i][j] for i in I) <= recursos[j] for j in J)

# Projetos não conduzidos simultaneamente
m.addConstr(x[1] + x[3] <= 1, "c02")

# Projetos conduzidos simultaneamente
m.addConstr(x[0] - x[5] == 0, "c03")

m.ModelSense = GRB.MAXIMIZE
m.update()
m.write("Modelo_Investimento-b.lp")
m.optimize()

# Saída...
for i in range(projetos):
    print(f"Projeto {i+1} -> Executar (1-sim, 0-não): {x[i].x}")


### Caso C


In [ ]:
# Modelo
m = gp.Model("Investimento-c")

# Variáveis de decisão
x = m.addVars (I, vtype = GRB.BINARY, obj = lucro, name = "x")
s = m.addVars (I, vtype = GRB.CONTINUOUS, lb = 0.0, obj = 0.0, name = "s")

# Restrições
# Respeitar a disponibilidade de recursos - Ano 0
m.addConstr(s[0] + gp.quicksum(x[i]*desembolso[i][0] for i in I) == recursos[0])

# Respeitar a disponibilidade de recursos - Anos 1, 2 e 3
m.addConstrs(s[j] + gp.quicksum(x[i]*desembolso[i][j] for i in I) == recursos[j] + s[j-1] for j in J if j>0)

m.ModelSense = GRB.MAXIMIZE
m.update()
m.write("Modelo_Investimento-c.lp")
m.optimize()

# Saída...
for i in range(projetos):
    print(f"Projeto {i+1} -> Executar (1-sim, 0-não): {x[i].x}")

### Caso D


In [ ]:
# Modelo
m = gp.Model("Investimento-d")

# Variáveis de decisão
x = m.addVars (I, vtype = GRB.CONTINUOUS, lb = 0.0, ub = 1.0, obj = lucro, name = "x")

# Restrições
# Respeitar a disponibilidade de recursos
m.addConstrs(gp.quicksum(x[i]*desembolso[i][j] for i in I) <= recursos[j] for j in J)

m.ModelSense = GRB.MAXIMIZE
m.update()
m.write("Modelo_Investimento-d.lp")
m.optimize()

# Saída...
for i in range(projetos):
    print(f"Projeto {i+1} -> Executar (1-sim, 0-não): {x[i].x}")

### Caso E


In [ ]:
# Nesta versão do problema foi admitido que qualquer fração do projeto possa ser realizada em qualquer ano,
# e que o desembolso em cada ano é proporcional à fração realizada do projeto.

# Modelo
m = gp.Model("Investimento-e")

# Variáveis de decisão
x = m.addVars (I, vtype = GRB.BINARY, obj = lucro, name = "x")
y = m.addVars (I, J, vtype = GRB.CONTINUOUS, lb = 0.0, ub = 1.0, obj = 0.0, name = "y")

# Restrições
# Respeitar a disponibilidade de recursos
m.addConstrs(gp.quicksum(y[i,j]*custo_total[i] for i in I) <= recursos[j] for j in J)

# As frações devem ser igual a 0 ou 1
m.addConstrs(gp.quicksum(y[i,j] for j in J) == x[i] for i in I)

m.ModelSense = GRB.MAXIMIZE
m.update()
m.write("Modelo_Investimento-e.lp")
m.optimize()

# Saída...
for i in range(projetos):
    print(f"Projeto {i+1} -> Executar (1-sim, 0-não): {x[i].x}")